In [ ]:
!pip install transformers torch

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

chat_history_ids = None

while True:
    user_input = input("You: ").strip().lower()

    if user_input in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break

    new_input = tokenizer(user_input + tokenizer.eos_token, return_tensors='pt')
    new_input_ids = new_input["input_ids"]
    attention_mask = new_input["attention_mask"]

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        attention_mask = torch.cat(
            [torch.ones(chat_history_ids.shape, dtype=torch.long), attention_mask],
            dim=-1
        )
    else:
        bot_input_ids = new_input_ids

    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=40,
        top_p=0.85,
        temperature=0.6,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: what is ai
Chatbot: AI is the name of the AI.
You: who introduce python
Chatbot: The creator introduced Python to us, so we are the creators of Python.
You: exit
Chatbot: Goodbye! 👋


#Output

Chatbot: Hello! I am your AI assistant.

You: what is ai

Chatbot: Artificial Intelligence is the simulation of human intelligence.

You: who created python

Chatbot: Python was created by Guido van Rossum.

You: exit

Chatbot: Goodbye!

Introduction

This project builds a chatbot using Hugging Face Transformers. It uses a pre-trained DialoGPT model to generate conversational responses.

🔹 Objective

To create an interactive chatbot that communicates with users using natural language processing.

🔹 Model Used

The chatbot uses the DialoGPT model from Hugging Face, which is trained on large conversational datasets.

🔹 Working Flow
User Input → Tokenization → Model → Response → Output → Loop
🔹 Conclusion

The chatbot successfully generates responses using a transformer-based model and maintains conversation flow without hardcoding.